# ML Model Training

This notebook mirrors the **Model Training** guide at [learnmlops.ops4life.com/guides/ml-model-training/](https://learnmlops.ops4life.com/guides/ml-model-training/).

Train a Logistic Regression classifier on the preprocessed employee attrition dataset, track the experiment with MLflow, and log the model artifact to the registry.

**What we'll cover:**
1. Load training data
2. Configure MLflow
3. Build a preprocessing pipeline
4. Train and log the experiment
5. Inspect results

In [ ]:
!pip install mlflow --quiet

import os
MLFLOW_URI = "file:///tmp/mlruns" if "COLAB_RELEASE_TAG" in os.environ else "http://mlflow:5000"


In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression   # Fast, interpretable baseline; good first model
from sklearn.preprocessing import StandardScaler       # Normalises features to zero mean, unit variance
from sklearn.pipeline import Pipeline   # Chains preprocessing + model into one object to prevent leakage
from sklearn.metrics import (
    accuracy_score,         # Fraction of correct predictions
    f1_score,               # Harmonic mean of precision and recall — better than accuracy for imbalanced data
    roc_auc_score,          # Area under ROC curve — threshold-independent discrimination measure
    classification_report,  # Per-class precision, recall, F1, support table
)
import mlflow              # Experiment tracking
import mlflow.sklearn      # Sklearn model serialisation and logging

# Point MLflow at the tracking server and group this notebook's runs
# under the 'employee-attrition' experiment.
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment('employee-attrition')


## Step 1: Load training data

In [ ]:
train_df = pd.read_csv('/workspace/pipeline-data/train.csv')
test_df  = pd.read_csv('/workspace/pipeline-data/test.csv')

X_train = train_df.drop(columns=['Attrition'])
y_train = train_df['Attrition']
X_test  = test_df.drop(columns=['Attrition'])
y_test  = test_df['Attrition']

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Attrition rate — Train: {y_train.mean():.1%}  Test: {y_test.mean():.1%}')

## Step 2: Configure MLflow

In [ ]:
# Verify the MLflow tracking server is reachable before starting a run.
# search_experiments() lists all experiments on the server — a lightweight call
# that will throw a connection error immediately if the server is down.
# Better to fail here than to train for 10 minutes and lose the run.
client = mlflow.tracking.MlflowClient()
experiments = client.search_experiments()
print(f'MLflow server: {mlflow.get_tracking_uri()}')
print(f'Existing experiments: {[e.name for e in experiments]}')


## Step 3: Build a preprocessing pipeline

In [ ]:
def build_pipeline(C=1.0, solver='lbfgs', max_iter=1000):
    """Return a sklearn Pipeline with StandardScaler + LogisticRegression.

    Args:
        C:        Inverse regularisation strength — smaller C = stronger regularisation.
                  C=1.0 is the sklearn default; decrease if overfitting (e.g. C=0.1).
        solver:   Optimisation algorithm. 'lbfgs' is efficient for small/medium datasets
                  and supports L2 regularisation. Use 'saga' for L1 or large datasets.
        max_iter: Maximum number of solver iterations. Increase if you get a
                  ConvergenceWarning during fit.
    """
    return Pipeline([
        # Step 1: StandardScaler normalises each feature to mean=0, std=1.
        # WHY: Logistic Regression uses gradient descent — features with large ranges
        # (e.g. MonthlyIncome: 1500–20000) dominate those with small ranges
        # (e.g. AgeGroup: 1–4). Scaling puts all features on equal footing and makes
        # the solver converge faster and more stably.
        # IMPORTANT: scaler.fit() is called only on training data. The test set is
        # only transformed (not fit) — this prevents information from the test
        # distribution leaking into the training process.
        ('scaler', StandardScaler()),

        # Step 2: Logistic Regression classifier.
        # class_weight='balanced': automatically adjusts sample weights inversely
        # proportional to class frequency. With ~16% attrition, without this the
        # model learns to predict "No attrition" 84% of the time and still looks
        # accurate — but it misses all the employees who actually leave.
        # class_weight='balanced' forces the model to pay equal attention to both classes.
        ('clf', LogisticRegression(
            C=C, solver=solver, max_iter=max_iter,
            class_weight='balanced',
            random_state=42,   # Ensures reproducible training across identical runs
        )),
    ])


A `Pipeline` chains preprocessing and model steps into a single object. This prevents data leakage: the scaler's `fit` is called only on training data, then `transform` is applied to both train and test. Wrapping the pipeline in MLflow gives us reproducible experiments — every run records its hyperparameters alongside its metrics.

## Step 4: Train and log the experiment

In [ ]:
with mlflow.start_run(run_name='lr-baseline') as run:
    C, solver, max_iter = 1.0, 'lbfgs', 1000

    # Log hyperparameters — values you SET before training.
    # These are searchable in the MLflow UI and used to reproduce the run.
    mlflow.log_param('C', C)
    mlflow.log_param('solver', solver)
    mlflow.log_param('max_iter', max_iter)
    mlflow.log_param('class_weight', 'balanced')

    pipeline = build_pipeline(C=C, solver=solver, max_iter=max_iter)

    # pipeline.fit() calls scaler.fit_transform(X_train) then clf.fit(X_scaled, y_train).
    # The Pipeline ensures the scaler never sees X_test during fitting.
    pipeline.fit(X_train, y_train)

    # predict() returns binary class labels (0 or 1) using a 0.5 probability threshold.
    y_pred = pipeline.predict(X_test)

    # predict_proba() returns a (n_samples, 2) array: [:, 0] = P(No attrition), [:, 1] = P(attrition).
    # We take column 1 — the probability of the positive class (attrition=1).
    # ROC-AUC uses probabilities (not binary labels) to evaluate discrimination
    # across all possible thresholds.
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred)        # F1 on the positive class (attrition=1)
    auc = roc_auc_score(y_test, y_prob)   # AUC=0.5 is random; AUC=1.0 is perfect

    # Log evaluation metrics — values COMPUTED after training.
    mlflow.log_metric('accuracy', acc)
    mlflow.log_metric('f1',       f1)
    mlflow.log_metric('roc_auc',  auc)

    # Log and register the model artifact.
    # 'model' is the artifact subdirectory path within the run's artifact store.
    # registered_model_name creates/updates a named model in the MLflow registry,
    # making it available for staging and production transitions in promote.ipynb.
    mlflow.sklearn.log_model(pipeline, 'model', registered_model_name='attrition-classifier')

    run_id = run.info.run_id
    print(f'Run ID:   {run_id}')
    print(f'Accuracy: {acc:.4f}')
    print(f'F1 score: {f1:.4f}')
    print(f'ROC-AUC:  {auc:.4f}')


Open the MLflow UI at [mlflow.learnmlops.ops4life.com](https://mlflow.learnmlops.ops4life.com) and navigate to the **employee-attrition** experiment to see the run. The logged model appears in the **Models** tab as `attrition-classifier`.

## Step 5: Inspect results

In [ ]:
print(classification_report(y_test, y_pred, target_names=['Stayed', 'Left']))

## Next Steps

- Tune hyperparameters with Optuna: `02-pipeline/ml-model-training/tune.ipynb`
- Full guide: [learnmlops.ops4life.com/guides/ml-model-training/](https://learnmlops.ops4life.com/guides/ml-model-training/)